
# Amazon ML Challenge 2026 — Stage 2
## SageMaker Studio: Script Detection + Indic → Roman Transliteration

This notebook is adapted for **Amazon SageMaker Studio / Jupyter** and reads the Stage-1 basic-cleaned `.tsv` files directly from **Amazon S3**.

### S3 configuration
Set `S3_INPUT_PREFIX` and `S3_OUTPUT_PREFIX` in the configuration cell below.

Recommended layout in the **same S3 bucket** as SageMaker Studio:

```text
s3://YOUR-BUCKET/amazon-ml-2026/cleaned_stage1/
    train_source1_basic_cleaned.tsv
    train_source2_basic_cleaned.tsv
    train_source3_basic_cleaned.tsv
    train_ground_truth.tsv
    test_source1_basic_cleaned.tsv
    test_source2_basic_cleaned.tsv
    test_source3_basic_cleaned.tsv

s3://YOUR-BUCKET/amazon-ml-2026/stage2_romanized/
    ... outputs ...
```

The pipeline keeps the Stage-1 columns and adds script/romanization columns. It processes source files in **100,000-row chunks** so the entire multi-million-row dataset is not loaded into RAM at once.


In [ ]:

# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================

# IMPORTANT:
# Replace YOUR-BUCKET with the actual S3 bucket name.
# Replace the prefixes below if your files are stored elsewhere.
#
# Example:
# S3_INPUT_PREFIX = "s3://my-sagemaker-bucket/amazon-ml-2026/cleaned_stage1/"
# S3_OUTPUT_PREFIX = "s3://my-sagemaker-bucket/amazon-ml-2026/stage2_romanized/"

S3_INPUT_PREFIX = "s3://YOUR-BUCKET/amazon-ml-2026/cleaned_stage1/"
S3_OUTPUT_PREFIX = "s3://YOUR-BUCKET/amazon-ml-2026/stage2_romanized/"

# Number of rows processed at a time.
# Your previous Colab pipeline used 100,000, so we keep that here.
CHUNK_SIZE = 100_000

# Set True to process training + test source files.
PROCESS_TRAIN = True
PROCESS_TEST = True

# Ground truth is copied unchanged; it is not transliterated.
COPY_GROUND_TRUTH = True

# ------------------------------------------------------------
# File names expected inside S3_INPUT_PREFIX
# ------------------------------------------------------------
TRAIN_FILES = {
    "train_source1": "train_source1_basic_cleaned.tsv",
    "train_source2": "train_source2_basic_cleaned.tsv",
    "train_source3": "train_source3_basic_cleaned.tsv",
}

TEST_FILES = {
    "test_source1": "test_source1_basic_cleaned.tsv",
    "test_source2": "test_source2_basic_cleaned.tsv",
    "test_source3": "test_source3_basic_cleaned.tsv",
}

GROUND_TRUTH_FILE = "train_ground_truth.tsv"

print("S3 input :", S3_INPUT_PREFIX)
print("S3 output:", S3_OUTPUT_PREFIX)
print("Chunk size:", f"{CHUNK_SIZE:,}")


In [ ]:

# ============================================================
# CELL 2 — INSTALL DEPENDENCIES
# ============================================================

# Install in the current SageMaker Jupyter kernel.
# %pip is preferred over !pip so the package goes into the
# notebook's active Python environment.

%pip install -q -U indic-transliteration s3fs tqdm

print("✓ Dependencies installed")


In [ ]:

# ============================================================
# CELL 3 — IMPORTS + AWS/S3 ACCESS TEST
# ============================================================

import os
import gc
import re
import shutil
import unicodedata
from collections import Counter

import boto3
import pandas as pd
from tqdm.auto import tqdm
from indic_transliteration import sanscript

# Parse S3 URIs
from urllib.parse import urlparse


def parse_s3_uri(uri):
    """Return (bucket, key_prefix) from an s3:// URI."""
    parsed = urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"Invalid S3 URI: {uri}")
    return parsed.netloc, parsed.path.lstrip("/")


input_bucket, input_prefix = parse_s3_uri(S3_INPUT_PREFIX)
output_bucket, output_prefix = parse_s3_uri(S3_OUTPUT_PREFIX)

# Uses the SageMaker notebook's IAM execution role / AWS credentials.
s3 = boto3.client("s3")

# Verify access to the input prefix.
print("Checking S3 access...")
response = s3.list_objects_v2(
    Bucket=input_bucket,
    Prefix=input_prefix,
    MaxKeys=10,
)

print(f"✓ Accessible bucket: {input_bucket}")
print(f"✓ Input prefix      : {input_prefix}")
print(f"✓ Objects found (first page): {response.get('KeyCount', 0)}")


In [ ]:

# ============================================================
# CELL 4 — SCRIPT DEFINITIONS
# ============================================================

SCRIPT_RANGES = {
    "Devanagari": [(0x0900, 0x097F)],
    "Bengali": [(0x0980, 0x09FF)],
    "Gurmukhi": [(0x0A00, 0x0A7F)],
    "Gujarati": [(0x0A80, 0x0AFF)],
    "Oriya": [(0x0B00, 0x0B7F)],
    "Tamil": [(0x0B80, 0x0BFF)],
    "Telugu": [(0x0C00, 0x0C7F)],
    "Kannada": [(0x0C80, 0x0CFF)],
    "Malayalam": [(0x0D00, 0x0D7F)],
    "Sinhala": [(0x0D80, 0x0DFF)],
}

SCRIPT_TO_SCHEME = {
    "Devanagari": sanscript.DEVANAGARI,
    "Bengali": sanscript.BENGALI,
    "Gurmukhi": sanscript.GURMUKHI,
    "Gujarati": sanscript.GUJARATI,
    "Oriya": sanscript.ORIYA,
    "Tamil": sanscript.TAMIL,
    "Telugu": sanscript.TELUGU,
    "Kannada": sanscript.KANNADA,
    "Malayalam": sanscript.MALAYALAM,
}


def get_char_script(ch):
    """Detect the meaningful script of one character."""
    code = ord(ch)

    # Check script ranges FIRST so Indic vowel signs/marks remain
    # attached to the relevant script.
    for script, ranges in SCRIPT_RANGES.items():
        for start, end in ranges:
            if start <= code <= end:
                return script

    category = unicodedata.category(ch)

    # Generic combining marks outside recognized script ranges.
    if category.startswith("M"):
        return None

    try:
        name = unicodedata.name(ch, "")
    except Exception:
        name = ""

    if "LATIN" in name:
        return "Latin"

    # Punctuation, digits, symbols and control chars do not define
    # the script of a text field.
    if (
        category.startswith("P")
        or category.startswith("N")
        or category.startswith("S")
        or category.startswith("C")
    ):
        return None

    return None


def detect_script(text):
    """Detect dominant script; return Mixed only when appropriate."""
    if pd.isna(text):
        return "Empty"

    text = str(text)
    if not text.strip():
        return "Empty"

    counts = Counter()
    for ch in text:
        script = get_char_script(ch)
        if script is not None:
            counts[script] += 1

    if not counts:
        return "Other"

    if len(counts) == 1:
        return next(iter(counts))

    ordered = counts.most_common()
    dominant_script, dominant_count = ordered[0]
    second_count = ordered[1][1]
    total = sum(counts.values())
    dominant_ratio = dominant_count / total

    # Do not call a record Mixed because of a few stray characters.
    if second_count <= 2:
        return dominant_script
    if dominant_ratio >= 0.90 and second_count <= 5:
        return dominant_script

    return "Mixed"


def transliterate_text(text):
    """Indic → Roman using Harvard-Kyoto; Latin text stays unchanged."""
    if pd.isna(text):
        return pd.NA

    text = str(text)
    if not text.strip():
        return text

    script = detect_script(text)

    # Latin-only: leave untouched.
    if script == "Latin":
        return text

    # Pure supported Indic script.
    if script in SCRIPT_TO_SCHEME:
        return sanscript.transliterate(
            text,
            SCRIPT_TO_SCHEME[script],
            sanscript.HK,
        )

    # Mixed scripts: transliterate only the Indic runs.
    if script == "Mixed":
        output = []
        buffer = []
        current_script = None

        def flush():
            nonlocal buffer, current_script
            if not buffer:
                return

            chunk = "".join(buffer)
            if current_script in SCRIPT_TO_SCHEME:
                chunk = sanscript.transliterate(
                    chunk,
                    SCRIPT_TO_SCHEME[current_script],
                    sanscript.HK,
                )

            output.append(chunk)
            buffer = []

        for ch in text:
            char_script = get_char_script(ch)

            if char_script in SCRIPT_TO_SCHEME:
                if current_script != char_script:
                    flush()
                    current_script = char_script
                buffer.append(ch)
            else:
                if current_script != char_script:
                    flush()
                    current_script = char_script
                buffer.append(ch)

        flush()
        return "".join(output)

    # Unsupported/unknown: preserve the input rather than inventing text.
    return text


def make_roman_search(text):
    """Create the simple Roman representation used in later matching."""
    if pd.isna(text):
        return pd.NA

    text = str(text)
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)

    # Remove combining marks/diacritics introduced by Romanization.
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = " ".join(text.split())
    return text

print("✓ Script detection and transliteration functions loaded")


In [ ]:

# ============================================================
# CELL 5 — QUICK REAL-DATA TEST (5,000 ROWS)
# ============================================================
# Run this before the full millions-row job.
# It reads only 5,000 rows from each source file directly from S3.

TEST_ROWS = 5_000


def s3_path(filename):
    return f"s3://{input_bucket}/{input_prefix.rstrip('/')}/{filename}"


sample_results = {}

for name, filename in {**TRAIN_FILES, **TEST_FILES}.items():

    path = s3_path(filename)

    print("\n" + "=" * 90)
    print(f"TESTING: {name}")
    print(path)
    print("=" * 90)

    df = pd.read_csv(
        path,
        sep="\t",
        encoding="utf-8",
        dtype="string",
        nrows=TEST_ROWS,
    )

    df["business_name_script"] = (
        df["business_name_basic"].map(detect_script)
    )

    df["business_address_script"] = (
        df["business_address_basic"].map(detect_script)
    )

    df["business_name_roman"] = (
        df["business_name_basic"].map(transliterate_text)
    )

    df["business_name_roman_search"] = (
        df["business_name_roman"].map(make_roman_search)
    )

    df["business_address_roman"] = (
        df["business_address_basic"].map(transliterate_text)
    )

    df["business_address_roman_search"] = (
        df["business_address_roman"].map(make_roman_search)
    )

    sample_results[name] = df

    print("\nBusiness-name scripts:")
    print(df["business_name_script"].value_counts().to_string())

    print("\nBusiness-address scripts:")
    print(df["business_address_script"].value_counts().to_string())

# Show actual non-Latin examples.
print("\n" + "=" * 90)
print("NON-LATIN / MIXED NAME SAMPLE")
print("=" * 90)

for name, df in sample_results.items():
    filtered = df[
        ~df["business_name_script"].isin(["Latin", "Empty", "Other"])
    ]

    if not filtered.empty:
        print(f"\n{name}")
        display(
            filtered[[
                "entity_id",
                "business_name_basic",
                "business_name_script",
                "business_name_roman",
                "business_name_roman_search",
                "country",
            ]].head(10)
        )


In [ ]:

# ============================================================
# CELL 6 — FULL S3 PROCESSING
# ============================================================
# Important:
# - Processes one source file at a time.
# - Reads 100,000 rows per chunk.
# - Appends to a LOCAL temporary TSV file.
# - Uploads the completed file to S3.
# - Deletes the local temporary file before moving to the next source.
#
# This avoids keeping millions of rows in RAM simultaneously.
# Make sure the SageMaker Studio storage volume has enough free space
# for the largest intermediate output file.
# ============================================================

LOCAL_WORK_DIR = "/tmp/amazon_ml_stage2"
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)


def upload_file_to_s3(local_path, bucket, key):
    """Upload a completed local file to S3."""
    s3.upload_file(local_path, bucket, key)
    print(f"✓ Uploaded: s3://{bucket}/{key}")


def process_full_source(dataset_name, filename):
    """Process one Stage-1 TSV from S3 into a Stage-2 TSV on S3."""

    input_path = s3_path(filename)

    output_filename = filename.replace(
        "_basic_cleaned.tsv",
        "_stage2_romanized.tsv",
    )

    output_key = f"{output_prefix.rstrip('/')}/{output_filename}"
    local_output = os.path.join(
        LOCAL_WORK_DIR,
        output_filename,
    )

    # Start clean for this output.
    if os.path.exists(local_output):
        os.remove(local_output)

    print("\n" + "=" * 100)
    print(f"PROCESSING: {dataset_name}")
    print("=" * 100)
    print("Input :", input_path)
    print("Output:", f"s3://{output_bucket}/{output_key}")
    print("Chunk :", f"{CHUNK_SIZE:,} rows")

    first_chunk = True
    total_rows = 0
    name_counts = Counter()
    address_counts = Counter()

    reader = pd.read_csv(
        input_path,
        sep="\t",
        encoding="utf-8",
        dtype="string",
        chunksize=CHUNK_SIZE,
    )

    try:
        for chunk_number, df in enumerate(reader, start=1):

            print(
                f"\nChunk {chunk_number:,} | rows={len(df):,}"
            )

            required = {
                "entity_id",
                "business_name",
                "business_address",
                "country",
                "business_name_original",
                "business_name_basic",
                "business_address_original",
                "business_address_basic",
            }

            missing = sorted(required - set(df.columns))

            if missing:
                raise ValueError(
                    f"{dataset_name}: missing required columns: {missing}"
                )

            # ------------------------------------------------
            # SCRIPT DETECTION
            # ------------------------------------------------
            df["business_name_script"] = [
                detect_script(v)
                for v in tqdm(
                    df["business_name_basic"],
                    desc="Name script",
                    leave=False,
                )
            ]

            df["business_address_script"] = [
                detect_script(v)
                for v in tqdm(
                    df["business_address_basic"],
                    desc="Address script",
                    leave=False,
                )
            ]

            name_counts.update(
                df["business_name_script"].value_counts().to_dict()
            )
            address_counts.update(
                df["business_address_script"].value_counts().to_dict()
            )

            # ------------------------------------------------
            # TRANSLITERATION
            # ------------------------------------------------
            df["business_name_roman"] = [
                transliterate_text(v)
                for v in tqdm(
                    df["business_name_basic"],
                    desc="Name → Roman",
                    leave=False,
                )
            ]

            df["business_name_roman_search"] = (
                df["business_name_roman"].map(make_roman_search)
            )

            df["business_address_roman"] = [
                transliterate_text(v)
                for v in tqdm(
                    df["business_address_basic"],
                    desc="Address → Roman",
                    leave=False,
                )
            ]

            df["business_address_roman_search"] = (
                df["business_address_roman"].map(make_roman_search)
            )

            # ------------------------------------------------
            # APPEND TO LOCAL OUTPUT
            # ------------------------------------------------
            df.to_csv(
                local_output,
                sep="\t",
                encoding="utf-8",
                index=False,
                mode="w" if first_chunk else "a",
                header=first_chunk,
            )

            first_chunk = False
            total_rows += len(df)

            print(f"✓ Written {total_rows:,} rows locally")

            del df
            gc.collect()

        # ----------------------------------------------------
        # UPLOAD COMPLETED FILE TO S3
        # ----------------------------------------------------
        upload_file_to_s3(
            local_output,
            output_bucket,
            output_key,
        )

    finally:
        # Always clean local storage even if processing fails.
        if os.path.exists(local_output):
            os.remove(local_output)

    print(f"\n✓ {dataset_name} COMPLETE")
    print(f"Total rows: {total_rows:,}")

    print("\nBusiness-name scripts:")
    for script, count in name_counts.most_common():
        print(f"  {script:<15} {count:,}")

    print("\nBusiness-address scripts:")
    for script, count in address_counts.most_common():
        print(f"  {script:<15} {count:,}")


# ------------------------------------------------------------
# PROCESS TRAINING SOURCES
# ------------------------------------------------------------
if PROCESS_TRAIN:
    for name, filename in TRAIN_FILES.items():
        process_full_source(name, filename)


# ------------------------------------------------------------
# COPY GROUND TRUTH UNCHANGED
# ------------------------------------------------------------
if PROCESS_TRAIN and COPY_GROUND_TRUTH:

    source_key = f"{input_prefix.rstrip('/')}/{GROUND_TRUTH_FILE}"
    target_key = f"{output_prefix.rstrip('/')}/{GROUND_TRUTH_FILE}"

    print("\nCopying ground truth unchanged...")

    s3.copy_object(
        Bucket=output_bucket,
        Key=target_key,
        CopySource={
            "Bucket": input_bucket,
            "Key": source_key,
        },
    )

    print(
        f"✓ Ground truth copied to s3://{output_bucket}/{target_key}"
    )


# ------------------------------------------------------------
# PROCESS TEST SOURCES
# ------------------------------------------------------------
if PROCESS_TEST:
    for name, filename in TEST_FILES.items():
        process_full_source(name, filename)


print("\n" + "=" * 100)
print("STAGE 2 FULL PROCESSING COMPLETE")
print("=" * 100)
print("Output prefix:")
print(f"s3://{output_bucket}/{output_prefix}")


In [ ]:

# ============================================================
# CELL 7 — LIST FINAL S3 OUTPUTS
# ============================================================

print("=" * 100)
print("FINAL STAGE-2 OUTPUT DIRECTORY")
print("=" * 100)

paginator = s3.get_paginator("list_objects_v2")

found = False

total_bytes = 0

for page in paginator.paginate(
    Bucket=output_bucket,
    Prefix=output_prefix,
):

    for obj in page.get("Contents", []):

        found = True
        total_bytes += obj["Size"]

        size_gb = obj["Size"] / (1024 ** 3)

        print(
            f"{obj['Key']:<80}"
            f"{size_gb:>8.2f} GB"
        )

if not found:
    print("No output files found.")
else:
    print("\nTotal output size:", f"{total_bytes / (1024**3):.2f} GB")

print(
    "\nS3 output URI:"
)
print(
    f"s3://{output_bucket}/{output_prefix}"
)
